# Kolmogorov velocity dataset: 256 -> 64, 100k samples

This notebook generates a larger chaotic Kolmogorov-flow-like dataset. It simulates forced 2D incompressible Navier-Stokes on a `256x256` grid, stores recovered velocity fields average-pooled to `64x64`, and then writes exact split shards: `80_000` train, `10_000` validation, `10_000` test.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess

REPO_URL = "https://github.com/SadreevAmir/DL_final_project.git"
REPO_DIR = "kolmogorov_100k_repo"

subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd {REPO_DIR}
!pip install -q -r requirements.txt

## Generate raw chunks

Raw chunks are temporary. The next cell will rewrite them into exact train/val/test split shards and can delete the raw chunks afterwards.

In [ ]:
from pathlib import Path
from google.colab import files
from IPython.display import display
from PIL import Image

from kolmogorov_dataset import KolmogorovConfig, generate_dataset

raw_dir = Path("data/kolmogorov_velocity_256_to_64_100k_raw")

config = KolmogorovConfig(
    output_dir=str(raw_dir),
    total_images=100_000,
    grid_size=256,
    save_grid_size=64,
    num_trajectories=2_000,
    max_trajectories=3_500,
    snapshots_per_trajectory=50,
    burn_in_steps=5_000,
    save_interval=20,
    chunk_size=5_000,
    sim_batch_size=64,
    dt=0.005,
    viscosity=3.0e-4,
    drag=0.025,
    forcing_amp=0.55,
    forcing_mode=4,
    param_mode="mixed",
    initial_amplitude=1.5,
    spectral_filter_scale=12.0,
    output_field="velocity",
    velocity_scale=1.0,
    vorticity_scale=6.0,
    vorticity_clip=60.0,
    dtype="float32",
    compress=False,
    seed=123,
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_threads=12,
    save_previews=True,
    preview_every_chunks=1,
    preview_count=32,
    preview_percentile=99.0,
    save_sequence_previews=True,
    sequence_preview_count=16,
    min_image_std=0.025,
    min_image_range=0.18,
)

def show_preview(path: Path):
    print(f"Preview: {path}")
    display(Image.open(path))

paths = generate_dataset(config, on_preview_saved=show_preview)
print(f"Saved {len(paths)} raw chunks in {raw_dir}")
files.download(str(raw_dir / "manifest.json"))

## Rewrite into exact split shards

This creates exact individual-snapshot splits. It shuffles rows within each raw chunk and fills the split quotas in order: train, val, test. The split is sample-based, not trajectory-based, because this final dataset is intended as independent images/snapshots.

In [ ]:
import json
import shutil
from collections import defaultdict

import numpy as np

split_dir = Path("data/kolmogorov_velocity_256_to_64_100k_splits")
split_counts = {"train": 80_000, "val": 10_000, "test": 10_000}
split_shard_size = 5_000
rng = np.random.default_rng(2026)
delete_raw_chunks_after_split = True

for split_name in split_counts:
    (split_dir / split_name).mkdir(parents=True, exist_ok=True)

keys = ["images", "trajectory_id", "snapshot_index", "step", "viscosity", "drag", "forcing_amp"]
buffers = {name: defaultdict(list) for name in split_counts}
buffer_counts = {name: 0 for name in split_counts}
written_counts = {name: 0 for name in split_counts}
shard_ids = {name: 0 for name in split_counts}
remaining = dict(split_counts)

def flush_split(split_name):
    if buffer_counts[split_name] == 0:
        return
    payload = {key: np.concatenate(buffers[split_name][key], axis=0) for key in keys}
    n = payload["images"].shape[0]
    payload["split"] = np.array([split_name] * n)
    path = split_dir / split_name / f"{split_name}_{shard_ids[split_name]:03d}.npz"
    np.savez(path, **payload)
    files.download(str(path))
    written_counts[split_name] += n
    shard_ids[split_name] += 1
    buffers[split_name].clear()
    buffer_counts[split_name] = 0

def append_to_split(split_name, arrays):
    offset = 0
    n = arrays["images"].shape[0]
    while offset < n:
        room = split_shard_size - buffer_counts[split_name]
        take = min(room, n - offset)
        for key in keys:
            buffers[split_name][key].append(arrays[key][offset:offset + take])
        buffer_counts[split_name] += take
        offset += take
        if buffer_counts[split_name] == split_shard_size:
            flush_split(split_name)

raw_chunks = sorted(raw_dir.glob("kolmogorov_chunk_*.npz"))
if not raw_chunks:
    raise RuntimeError(f"No raw chunks found in {raw_dir}")

current_split_order = ["train", "val", "test"]
for chunk_path in raw_chunks:
    with np.load(chunk_path) as chunk:
        n = chunk["images"].shape[0]
        perm = rng.permutation(n)
        offset = 0
        while offset < n and any(value > 0 for value in remaining.values()):
            split_name = next(name for name in current_split_order if remaining[name] > 0)
            take = min(remaining[split_name], n - offset)
            idx = perm[offset:offset + take]
            arrays = {key: chunk[key][idx] for key in keys}
            append_to_split(split_name, arrays)
            remaining[split_name] -= take
            offset += take

for split_name in split_counts:
    flush_split(split_name)

if any(value != 0 for value in remaining.values()):
    raise RuntimeError(f"Not enough samples to fill splits: {remaining}")

split_manifest = {
    "split_counts": split_counts,
    "written_counts": written_counts,
    "split_shard_size": split_shard_size,
    "source_raw_dir": str(raw_dir),
    "sample_shape": [2, 64, 64],
    "dtype": "float32",
    "split_strategy": "sample_based_after_chunkwise_shuffle",
    "field": "normalized_velocity",
    "channels": ["u_x", "u_y"],
}
(split_dir / "split_manifest.json").write_text(json.dumps(split_manifest, indent=2))
files.download(str(split_dir / "split_manifest.json"))

if delete_raw_chunks_after_split:
    for path in raw_chunks:
        path.unlink()

print(json.dumps(split_manifest, indent=2))

## Inspect one split shard

In [ ]:
import matplotlib.pyplot as plt

sample_shard = sorted((split_dir / "train").glob("train_*.npz"))[0]
data = np.load(sample_shard)
images = data["images"]
print(sample_shard, images.shape, images.dtype, float(images.min()), float(images.max()))

ux = images[:32, 0]
uy = images[:32, 1]
visual = 0.5 * (np.roll(uy, -1, axis=-1) - np.roll(uy, 1, axis=-1)) - 0.5 * (np.roll(ux, -1, axis=-2) - np.roll(ux, 1, axis=-2))
limit = max(1e-3, float(np.percentile(np.abs(visual), 99)))

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for ax, img in zip(axes.ravel(), visual):
    ax.imshow(img, cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.axis("off")
plt.tight_layout()